# Observational Causal Research

**Topic 05 · 2 lectures**

<hr>

# <center><a class="tocSkip"></center>
# <center>HONR 46400 — Evidence-Driven Research</center>
# <center>Professor: Davi Moreira</center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/notebooks/student/nb05_observational_causal_student.ipynb)

---

## 🧭 Inquiry & Claim Boundary

**Inquiry emphasis:** causal reasoning (causal kind, observational pathway). A
**causal question** asks what would change if you intervened, not just what goes
with what. Example: "does drinking coffee daily actually lengthen life, or do
coffee drinkers just happen to live longer?" This week you learn what it takes to
answer that honestly when you cannot run an experiment, and where the honest
answer stops at "associated with" instead of "because."

**Design pathway:** observational causal. **Observational** means you watch and
record without assigning anyone a treatment. Nature, or a rule, or a person's own
choice decided who got treated. This pathway can sometimes earn a causal claim,
but only through a stated argument that survives a skeptic, never by the size of
the comparison.

| | |
|---|---|
| **A claim this topic PERMITS** | "Under the identification argument I stated and probed, the estimated effect of [treatment] on [outcome] is [value] with uncertainty; here is the one assumption a skeptic would attack." |
| **A claim this topic does NOT permit** | "We controlled for everything, so this is the causal effect" when an unobserved confounder remains, or a difference-in-differences / instrumental-variables / regression-discontinuity label decorating a comparison whose key assumption was never stated or probed. |

**Where this sits in the course:** Week 6 of the design library. It builds on the
observational descriptive audit you defended in M4, and it develops M5, your
causal identification strategy (or your honest causal-language boundary), which
you submit at this week's Friday studio. M5 then hands off to the experimental
question that opens next.

*Provenance: RDSS ch. 16 'Observational: causal' + ch. 6-7 (model/inquiry; potential outcomes; SATE vs PATE) + the DeclareDesign observational-causal design library + rdss cliningsmith_etal | counterfactuals, estimand, confounding, selection into treatment, DAGs, selection-on-observables, DiD/IV/RDD as identification arguments, natural experiments, causal-language boundaries | seeded potential-outcomes simulation with a hidden confounder (seed 464) + a real natural experiment analyzed as a difference in means | translated (R to Python, honors non-quant audience)*

## Learning Objectives

By the end of this notebook, you will be able to:

1. State the **counterfactual** at the heart of any causal claim using potential
   outcomes Y(1) and Y(0), and name the **estimand** you are actually after.
2. Draw a **causal diagram** that shows a **confounder** opening a back door, and
   read when adjusting for observed variables identifies an effect and when it
   only pretends to.
3. Show, by seeded simulation, that a naive comparison misses the true effect
   under a hidden confounder, and that adjusting for an *observed* confounder can
   recover it.
4. Analyze a real **natural experiment** (a visa lottery) as a difference in means
   with uncertainty, and say exactly what its as-if-random assignment licenses.
5. Match **difference-in-differences**, **instrumental variables**, and
   **regression discontinuity** to the one untestable assumption each rests on,
   and name the **design mimicry** that fakes them.
6. Apply an identification argument, or an honest causal-language boundary, to
   your own project's M5 deliverable.

---

In [ ]:
# Setup — run this cell first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx  # for the causal diagrams; if missing locally: pip install networkx (pre-installed in Colab)

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (9, 5)

SEED = 464  # course number — keeps every simulation reproducible
rng = np.random.default_rng(SEED)

# Data loads: GitHub raw URL first (works in Colab), local repo path as fallback.
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2026F_evidence_driven_research_purdue_HONR464/main/notebooks/data/")

def load_course_data(filename):
    """Load a course dataset from GitHub, falling back to the local repo copy."""
    try:
        return pd.read_csv(DATA_URL + filename)
    except Exception:
        from pathlib import Path
        local = Path("notebooks/data") / filename
        if not local.exists():
            local = Path("../data") / filename
        return pd.read_csv(local)

print("✓ Setup complete — seed", SEED)

### 🤝 AI Research Partner

This week you work with **Gemini** as a research partner on the hardest word in
research: *because*. A good use looks like this. You commit your own answer first,
then ask Gemini to name a confounder you might have missed, explain why a
simulation behaved the way it did, or attack the assumption your causal claim
rests on. A bad use is letting it decide whether your design earns a causal claim,
or letting a fluent paragraph talk you past a hole in your argument.

**Never delegate these:** whether your design identifies a causal effect, which
confounder most threatens your comparison, and whether your finding earns
"because" or must stop at "associated with." Those are yours to declare and
defend. The full list of never-delegate decisions lives in
[`ai_resources/human_responsibility_checklist.md`](https://github.com/davi-moreira/2026F_evidence_driven_research_purdue_HONR464/blob/main/ai_resources/human_responsibility_checklist.md).

**Commit first, then ask.** Every AI block below asks you to write your own answer
before you open Gemini. Commit your work to git before a long AI session, so you
can always compare what you wrote against what the tool changed.

> **A question that often comes up here:** *"A causal question is hard. Why not let
> Gemini judge whether my design works?"* Because judging identification is the
> whole skill this week teaches, and it is exactly the kind of judgment an AI
> fakes best. A model will happily call your comparison a "natural experiment"
> because you used the words. Only you know how your treatment was really
> assigned, and that is what decides whether the word "because" is earned.

---

# Lecture 1

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

Here is a real pattern that shows up in health data again and again: **people who
drink coffee every day live longer, on average, than people who do not.** The
data are honest and the gap is real.

The headline writes itself: "coffee makes you live longer." Would you stake your
name on it? Commit to yes or no before we go further. Then defend it. On a piece
of paper, draw three boxes: *coffee*, *longevity*, and one more box for a trait
that might drive **both**. Draw the arrows. If your third box points to coffee
*and* to longevity, you have just drawn the reason the headline could be a trap.
Naming that third box, and the arrows, is the whole job of Lecture 1.

## 1. After Is Not Because: Counterfactuals and Confounders

**Guiding question:** *what would it actually take to say coffee lengthens life,
rather than that coffee drinkers happen to live longer?*

> *"Someone always tells me the patients who took the drug got better. Fine. So
> did the ones who rested, who were younger, who were richer. 'They got better
> after X' is not 'X worked.' It is a story missing its comparison. Show me the
> world without X, or admit you are guessing."*
> — a **skeptical peer** who refuses to confuse *after* with *because*

Start with the pill in your hand. Your headache fades an hour after you take it.
Did the pill work? The honest answer needs the world where you skipped the pill.
That missing world has a name.

- **Counterfactual**: the outcome that would have happened under the choice you did
  not make. Example: your headache an hour from now *if you had not* taken the
  pill. You observed the world where you took it. The counterfactual is the other
  world, and it never happened.

Write it once in symbols, because the notation pays for itself. The **treatment**
is the thing being tested, here daily coffee. The **outcome** is what you measure
afterward, here how long someone lives. For one person, Y(1) is the outcome with
the treatment and Y(0) is the outcome without it. The causal effect for that
person is the difference, Y(1) minus Y(0): their life with coffee minus their life
without it.

- **Estimand**: the exact quantity you are trying to learn, written down before you
  compute anything. Example: the *average* of Y(1) minus Y(0) across a group, the
  average causal effect of daily coffee on lifespan. Naming the estimand keeps you
  honest about what number would answer your question.

Now the catch that governs the whole field. The **fundamental problem of causal
inference** is that for any one person you only ever see *one* of the two worlds.
A person either drank coffee or did not. You never observe both their Y(1) and
their Y(0). No amount of staring at the life a coffee drinker actually lived
recovers the life they *would have* lived without coffee.

So you give up on the single person and go after the average across a group. That
turns the whole problem into one question: **is the no-coffee group a fair
stand-in for what the coffee group would have done without coffee?** Usually it is
not, and here is the villain that ruins it.

- **Confounder**: a third factor that pushes on *both* who gets the treatment and
  the outcome. Example: an active, health-conscious lifestyle. Health-conscious
  people are more likely to have a daily coffee routine, *and* they tend to live
  longer for reasons that have nothing to do with coffee. Diet, exercise, and
  checkups all ride along.

A confounder is exactly the third box from the puzzle. When it points to both
coffee and longevity, coffee drinkers look longer-lived even if coffee did
nothing at all, because the coffee group is stuffed with health-conscious people.
That is why "the treated did better" proves so little on its own.

> **A question that often comes up here:** *"My dataset is huge. Doesn't a big
> sample fix this?"* No, and this is the trap that catches earnest researchers.
> A bigger sample shrinks *noise*, the sample-to-sample wobble. It does nothing to
> a confounder, which is a *systematic* tilt in who got the treatment. A million
> coffee drinkers who are mostly health-conscious give you a very precise estimate
> of the wrong number. You will watch that happen with your own eyes below.

## 2. The Confounding DAG: Drawing the Back Door

**Guiding question:** *how do you draw the difference between a correlation that is
a trap and one that carries a cause?*

A picture settles arguments that words drag out. This one has a name.

- **Causal diagram**, also called a **DAG** (short for directed acyclic graph, a
  drawing of arrows where each arrow means "this directly influences that").
  Example: an arrow from *coffee* to *longevity* says coffee directly affects how
  long you live. This course calls it a **causal diagram** from here on, the same
  name your M5 uses.

Two more words let you read the diagram.

- **Selection into treatment**: the process that decided who got treated. Example:
  people *chose* their coffee habit, and health-conscious people chose it more
  often. Nobody flipped a coin. In observational research, selection into
  treatment is almost never random, and that is the root of the trouble.
- **Back door**: a sneaky path connecting treatment and outcome that runs *through*
  a confounder instead of through the real effect. Example: lifestyle points into
  *both* coffee and longevity, which links the two even when coffee does nothing.
  That shared path through lifestyle is a back door, and it manufactures correlation
  that is not cause.

Below, the left diagram is the trap, and the right one shows the two ways to shut
the back door. Draw it, then read it.

**🏠 Homework depth: run this on your own before the next lecture.**
**Before you ask:** in one sentence, list the confounders *you* think could drive
both coffee and longevity. Commit your own list first, so you can see whether the
tool finds one you missed, or invents one that does not belong.

> 💡 **Gemini Prompt:** "I am drawing a causal diagram for the claim that daily
> coffee lengthens life. Coffee is the treatment, longevity is the outcome. List
> the specific confounders, traits that plausibly raise *both* the chance of a
> daily coffee habit and lifespan, in a table with one column for the trait and
> one for the direction it would bias a naive coffee-versus-no-coffee comparison.
> Then, as a hostile reviewer, name the one trait on your list I am most likely
> wrong to include, and the one important confounder you suspect you left out."
>
> **After running, verify (counters *illusion of completeness*: a long, tidy list
> of confounders that quietly omits the one that matters most):**
> - [ ] Compare the list against the confounders you committed to above. If yours
>   is missing, the tidy output just failed the one test it looked complete on.
> - [ ] Map each trait it names to a back-door path in the diagram you draw below
>   (trait to coffee, trait to longevity). A trait with only one arrow is not a
>   confounder, so cross it off.
> - [ ] Decide *yourself* whether each trait plausibly moves longevity. The tool
>   lists names; only you can judge the mechanism.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Two causal diagrams: the confounding trap, and the two ways to close the back door.
# (networkx is imported once in the setup cell.)
def draw_dag(ax, edges, edge_colors, edge_labels, title, cut_edge=None):
    pos = {"Active\nlifestyle": (1.0, 2.0),   # the confounder, drawn on top
           "Daily\ncoffee":     (0.0, 0.7),   # the treatment
           "Longevity":          (2.0, 0.7)}   # the outcome
    G = nx.DiGraph(); G.add_edges_from(edges)
    node_colors = ["#F2C9A0" if n == "Active\nlifestyle" else "#DCE6F1" for n in G.nodes]
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, edgecolors="#333333",
                           linewidths=1.5, node_size=6000, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
    styles = ["dashed" if e == cut_edge else "solid" for e in edges]
    for e, c, s in zip(edges, edge_colors, styles):
        nx.draw_networkx_edges(G, pos, edgelist=[e], edge_color=c, style=s,
                               arrowstyle="-|>", arrowsize=20, width=2.4,
                               node_size=6000, ax=ax)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7.5,
                                 label_pos=0.5, ax=ax)
    ax.set_title(title, fontsize=10); ax.axis("off")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# LEFT: the trap. Lifestyle points to BOTH coffee and longevity (the back door).
trap_edges = [("Active\nlifestyle", "Daily\ncoffee"),
              ("Active\nlifestyle", "Longevity"),
              ("Daily\ncoffee", "Longevity")]
draw_dag(ax1, trap_edges,
         edge_colors=["#DD8452", "#DD8452", "#4C72B0"],
         edge_labels={trap_edges[0]: "back door", trap_edges[1]: "back door",
                      trap_edges[2]: "the effect\nyou want"},
         title="The trap: lifestyle points into BOTH coffee and longevity (a back door)")

# RIGHT: close the back door. Cut lifestyle -> coffee (randomize) OR condition on it.
fix_edges = [("Active\nlifestyle", "Daily\ncoffee"),
             ("Active\nlifestyle", "Longevity"),
             ("Daily\ncoffee", "Longevity")]
draw_dag(ax2, fix_edges,
         edge_colors=["#999999", "#DD8452", "#4C72B0"],
         edge_labels={fix_edges[0]: "cut by randomizing,\nor block by adjusting",
                      fix_edges[2]: "now the only link:\nthe real effect"},
         title="The fix: shut the back door (randomize, or adjust for lifestyle)",
         cut_edge=fix_edges[0])

plt.tight_layout(); plt.show()
print("✓ Two causal diagrams drawn.")
print("  Left: lifestyle feeds BOTH coffee and longevity, so the raw correlation")
print("        mixes the real effect with a back-door path. That is confounding.")
print("  Right: cut lifestyle -> coffee (a coin flip) or hold lifestyle fixed")
print("         (adjust for it), and the only path left is coffee -> longevity.")

**Reading the diagram.** In the left picture, two things connect coffee and
longevity. One is the arrow you care about, coffee to longevity. The other is a
back-door path riding through the confounder: lifestyle points into both coffee and
longevity, linking the two without coffee doing anything. A raw comparison of coffee
drinkers and non-drinkers sees the *sum* of both, so it overstates the effect of
coffee.

The right picture shows the only two escapes. **Randomize** the treatment, letting
a coin flip decide who drinks coffee, and the arrow from lifestyle to coffee
disappears, because chance ignores lifestyle. That is the experiment, and it is
next week's topic. Or, without an experiment, **adjust** for lifestyle: compare
coffee drinkers and non-drinkers *within* the same lifestyle group, so lifestyle
can no longer differ between the groups. Both moves shut the back door. The
simulation below shows the second one working, and shows exactly when it fails.

*In plain terms, the picture says a confounder gives correlation a second,
dishonest path, and you win only by cutting that path or holding the confounder
still.*

## 3. Watch a Naive Comparison Miss the Truth

**Guiding question:** *if a confounder is hiding in your data, how far off can a
plain comparison be, and can you fix it?*

Real data never hands you the true effect to check against. A simulation does. You
will build a small world where *you* set the true effect of coffee, then plant a
confounder, and watch a naive comparison miss the number you planted. Because you
built the world, you know the right answer, so you can grade every method against
it.

Here is the world. Half the people are health-conscious (the confounder). The
health-conscious drink daily coffee more often (70% of them do, versus 30% of
everyone else), so **selection into treatment is not random**. Coffee has a real
but modest effect on the health-and-longevity score you measure. You will set that
true effect yourself in the code.

### 🔮 Pause & Predict

You built coffee's true effect to be **+1.0 point** on the score. But coffee
drinkers are stuffed with health-conscious people, who score higher for reasons
unrelated to coffee. Before running the next cell, commit: when you compare the
average score of coffee drinkers against non-drinkers with no adjustment, will the
gap come out near +1.0, or bigger, or smaller? Write one number and one sentence
of reasoning. Do not peek at the output first. You are predicting a reveal.

### YOUR ANSWER HERE:

**My predicted naive gap (a number):**

**Bigger or smaller than the true +1.0, and why:**

---

### 🛠️ Run the Study: build the world, then compare naively

Run the cell below. It builds the confounded world, sets coffee's true effect to
+1.0, then reports the **naive** difference: average score of coffee drinkers
minus non-drinkers, with no adjustment. Read the gap before you read the next
markdown cell.

**🔴 Live in class: we run this one together.**
**Before you ask:** write one sentence predicting whether the naive gap will land
above or below the true +1.0 (you set a number last cell; now commit the reason).

> 💡 **Gemini Prompt:** "Here is a Python cell that builds a simulated world where
> health-conscious people both drink more coffee and live longer, sets coffee's
> true effect to +1.0, then prints the naive coffee-versus-no-coffee difference in
> a health score: [paste the next cell]. Explain, line by line, why the naive
> difference comes out larger than the +1.0 true effect the code planted. Then give
> me one independent way to confirm the naive number the cell printed, without
> rerunning the same code."
>
> **After running, verify (counters *confident fabrication*: a fluent line-by-line
> walkthrough can describe a number the code never produced):**
> - [ ] Confirm the naive gap Gemini quotes matches the value your cell actually
>   printed, not a number it guessed.
> - [ ] Check its explanation against the printed confounder split: are coffee
>   drinkers really more health-conscious than non-drinkers, by the two shares your
>   cell printed? If its story does not use those numbers, it is narrating, not verifying.
> - [ ] Run its "independent way to confirm" yourself. If it will not run or gives
>   a different gap, trust your printout.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Build a confounded world. A FRESH generator (seed 464) makes this cell
# reproducible on its own, regardless of any cell run before it.
sim_rng = np.random.default_rng(SEED)
N = 1500

# The hidden confounder: 1 = active / health-conscious lifestyle, 0 = not (~half each).
lifestyle = (sim_rng.random(N) < 0.5).astype(int)

# Selection into treatment is NOT random: the health-conscious drink coffee more.
p_coffee = np.where(lifestyle == 1, 0.70, 0.30)
coffee = (sim_rng.random(N) < p_coffee).astype(int)      # 1 = daily coffee (the treatment)

# Potential outcomes. TRUE effect of coffee is +1.0; lifestyle lifts the score by 6.
TRUE_EFFECT = 1.0
Y0 = 74.0 + 6.0 * lifestyle + sim_rng.normal(0, 2.0, N)  # score WITHOUT coffee
Y1 = Y0 + TRUE_EFFECT                                     # score WITH coffee
score = np.where(coffee == 1, Y1, Y0)                    # the score you actually observe

# The naive comparison: coffee drinkers vs non-drinkers, no adjustment.
naive_gap = score[coffee == 1].mean() - score[coffee == 0].mean()
true_sate = (Y1 - Y0).mean()
hc_coffee = lifestyle[coffee == 1].mean()
hc_nocoffee = lifestyle[coffee == 0].mean()

print(f"Coffee drinkers: {int(coffee.sum())}   Non-drinkers: {int((1 - coffee).sum())}")
print(f"Health-conscious share among coffee drinkers: {hc_coffee:.0%}")
print(f"Health-conscious share among non-drinkers   : {hc_nocoffee:.0%}")
print()
print(f"TRUE effect of coffee (you built it)   : {true_sate:+.2f} points")
print(f"NAIVE difference (coffee - no coffee)  : {naive_gap:+.2f} points")
print(f"The naive number overstates the truth by {naive_gap - true_sate:+.2f} points.")

In [ ]:
# Self-check: the naive gap is far above the true effect, driven by the confounder.
assert abs(true_sate - 1.0) < 1e-9, "the built-in true effect should be exactly +1.0"
assert 3.0 < naive_gap < 3.3, "naive gap drifted — is the seed 464 and the call order unchanged?"
assert hc_coffee > 0.6 and hc_nocoffee < 0.4, "coffee drinkers should be far more health-conscious"
print(f"✓ Self-check passed: true effect {true_sate:+.2f}, but the naive gap is "
      f"{naive_gap:+.2f}.\n  The confounder (lifestyle) inflates it by about "
      f"{naive_gap - true_sate:+.1f} points. That gap is bias, not noise.")

**Reading the evidence.** You planted a true effect of +1.0, but the naive
comparison reports about +3.1, roughly three times too big. Nothing is wrong with
the arithmetic. The coffee group is about 68% health-conscious against about 33%
for the non-drinkers, so the comparison mostly measures the *lifestyle* gap and
credits it to coffee. This is the back door from section 2, now in numbers. A
bigger sample would make +3.1 more precise and no less wrong.

So try the fix. Because you can *see* lifestyle in this simulated world, you can
adjust for it: compare coffee drinkers to non-drinkers *within* the health-conscious
group, then again *within* the not-health-conscious group, and average the two
honest within-group differences. This move has a name.

- **Selection on observables**: identifying an effect by adjusting for every
  confounder you can measure, so treated and untreated match on all of them.
  Example: comparing coffee drinkers and non-drinkers who share the same lifestyle.
  It works only for confounders you actually observe.

**Commit first:** before you run the adjustment, predict. When you compare coffee
drinkers and non-drinkers *within the same lifestyle group*, will the coffee
effect shrink back toward the true +1.0, or stay near +3.1? Write one sentence,
then run the cell.

In [ ]:
# Adjust for the observed confounder: compare WITHIN each lifestyle group, then average.
d_not = (score[(lifestyle == 0) & (coffee == 1)].mean()
         - score[(lifestyle == 0) & (coffee == 0)].mean())
d_hc = (score[(lifestyle == 1) & (coffee == 1)].mean()
        - score[(lifestyle == 1) & (coffee == 0)].mean())
n_not = int((lifestyle == 0).sum())
n_hc = int((lifestyle == 1).sum())
adjusted = (d_not * n_not + d_hc * n_hc) / (n_not + n_hc)

print("Coffee effect estimated WITHIN each lifestyle group:")
print(f"  not health-conscious (n={n_not}): {d_not:+.2f} points")
print(f"  health-conscious     (n={n_hc}): {d_hc:+.2f} points")
print(f"\nAdjusted effect (size-weighted average): {adjusted:+.2f} points")
print(f"True effect for comparison             : {true_sate:+.2f} points")

# Show the three side by side. TRUE is the reference; NAIVE is the biased one.
fig, ax = plt.subplots(figsize=(7.5, 5))
bars = ax.bar(["Naive\n(no adjustment)", "Adjusted\n(within lifestyle)"],
              [naive_gap, adjusted], color=["#DD8452", "#4C72B0"], edgecolor="white")
ax.axhline(true_sate, color="#C44E52", linestyle="--", linewidth=2,
           label=f"true effect = {true_sate:+.1f}")
ax.set_ylabel("Estimated effect of coffee (points)")
ax.set_title("Adjusting for the observed confounder pulls the estimate back to the truth")
ax.legend(loc="upper right")
for b, v in zip(bars, [naive_gap, adjusted]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.08, f"{v:+.2f}", ha="center", fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Self-check: adjustment lands near the truth; the naive estimate does not.
assert 0.8 < adjusted < 1.1, "adjusted estimate drifted — investigate the seed/call order"
assert adjusted < naive_gap, "adjusting should shrink the estimate toward the truth"
assert abs(adjusted - true_sate) < abs(naive_gap - true_sate), \
    "the adjusted estimate must be closer to the truth than the naive one"
print(f"✓ Self-check passed: the naive gap ({naive_gap:+.2f}) sits far above the truth "
      f"(+1.00),\n  but adjusting within lifestyle recovers about {adjusted:+.2f}. "
      "Same data, honest picture.")

### 🔍 Reading the Evidence: what identification needed

Adjusting for lifestyle worked, but read *why* before you celebrate. In the cell
below, answer three things. First: the adjusted estimate landed near the true
+1.0, but only because you could *see* lifestyle. What happens to your estimate if
lifestyle is real but never measured in your data? Name the number you would be
stuck reporting. Second: this is why "we controlled for everything" is a claim to
distrust. Explain, in one sentence, why controlling for the confounders you happen
to *observe* is not the same as identifying the effect. Third: name the single
variable whose absence would break identification here, and say how you would know
you were missing it.

### YOUR ANSWER HERE:

**What my estimate becomes if lifestyle is unmeasured (a number):**

**Why 'controlled for everything' is not identification:**

**The variable whose absence breaks identification, and how I would notice:**

---

---

### ⏸ In class through here

**You have reached the end of Lecture 1's in-class block.** Everything below
deepens the same ideas. Finish it on your own before the next lecture: the design
choice, the three-claim DAG practice, and the homework-depth Gemini prompt you met
above. Come back with your project's causal diagram and its most-feared confounder
drawn.

---

### ⚖️ Make a Design Choice: adjust, or stop at association?

*(Human-first: settle your own choice and its defense before you ask any AI.)*

Turn the lens on your own project. You want to claim that some treatment X affects
some outcome Y, but you cannot run an experiment. Choose one path and defend it in
a short paragraph:

- **A. Selection on observables.** You will adjust for a set of confounders you can
  measure, and argue that no important confounder is left unmeasured. Name the
  confounders you would adjust for, and the one you most fear you cannot measure.
- **B. Stop at association.** You cannot credibly rule out an unobserved confounder,
  so your honest claim is that X is *associated with* Y, with no causal reading. You
  will word every finding that way.

Choosing B is not a defeat. A precisely bounded associational claim is honest,
publishable research. A causal claim with an unmeasured confounder is a defect. In
the cell below, pick A or B and defend it.

### YOUR ANSWER HERE:

**My choice (A / B):**

**If A: the confounders I adjust for, and the one I most fear is unmeasured:**

**If B: the unobserved confounder I cannot rule out, so I stop at 'associated with':**

---

### 📝 Practice: draw the confounder, name what breaks identification

*(Human-first: reason through all three yourself before any AI. The DAG is the
graded skill.)*

Confounding is everywhere in STEM, not just health. For each claim below, name the
**confounder** that plausibly drives both the treatment and the outcome, sketch the
back-door path in words (the confounder pointing into both treatment and outcome),
and name the one
variable whose absence from the data would break identification. Answer in the cell
that follows.

- **A. Agronomy.** "Fields that got the new fertilizer had higher yields, so the
  fertilizer raised yields." (Think about which fields a farmer chooses to treat.)
- **B. Epidemiology.** "People who took the supplement had less heart disease, so
  the supplement protects the heart." (Think about who chooses supplements.)
- **C. Policy.** "Cities that adopted the job-training program saw employment rise,
  so the program created jobs." (Think about which cities adopt such programs.)

### YOUR ANSWER HERE:

**A (confounder / back-door path / variable whose absence breaks identification):**

**B (confounder / back-door path / variable whose absence breaks identification):**

**C (confounder / back-door path / variable whose absence breaks identification):**

---

You have drawn the back door and watched a confounder inflate a comparison by
three times. Lecture 2 turns to the rescue that does not require you to measure
every confounder: the moments when nature, or a rule, assigns treatment as-if at
random, and the honest boundary on what those moments let you claim.

---

---

# Lecture 2

### 🧩 Research Puzzle

*(Your research lead opens the lecture with this. Think it through and commit an
answer before we go further. No AI yet.)*

A scarce opportunity is handed out by **lottery**. Picture a limited number of
travel visas, and thousands of applicants, so a computer draws the winners at
random. Some people apply and win. Some apply and lose.

A researcher looking at this setup gets unusually excited. Before we go further,
answer two questions. First, *why* would a researcher call this lottery a gift for
studying the effect of the trip, when they could not ethically assign the trip
themselves? Second, name the one thing that, if it were true, would *ruin* the
logic and make the lottery useless for a fair comparison. Commit both answers.
Lecture 2 is about what that luck buys you, and what quietly takes it away.

## 4. When Nature Assigns: Natural Experiments

**Guiding question:** *when you cannot randomize, where might the world have
randomized for you, and can you defend that it really did?*

> *"I do not get to randomize whether a country goes to war or whether a person
> wins a visa. Most questions I care about forbid the experiment. So my whole craft
> is hunting for the moments where the world randomized FOR me, and being brutal
> about whether it truly did."*
> — the same **skeptical peer**, now working without an experiment

Lecture 1 ended in a hard place: if an important confounder is unmeasured,
adjustment cannot save you. Sometimes the world offers a way out.

- **Natural experiment**: a situation where a chance-like force outside anyone's
  research control decided who got treated. Example: a visa lottery decides who
  makes a trip; a birthday cutoff decides who starts school a year early. Nobody
  ran the study, but the assignment behaves like a coin flip.
- **As-if random**: assignment that was not deliberately randomized but is
  credibly *like* random, so treated and untreated are comparable. Example: among
  people who applied for a visa, a fair lottery makes winners and losers alike on
  everything except winning.

That last phrase is the whole engine. When assignment is as-if random, the
untreated group becomes an honest stand-in for the treated group's missing world,
with no back door for a confounder to sneak through. The lottery does for free what
Lecture 1 could not buy: it closes the back door without your having to measure a
single confounder. Below you analyze a real one.

## 5. A Real Natural Experiment: the Hajj Visa Lottery

**Guiding question:** *what, exactly, does an as-if-random lottery let you claim,
and over whom?*

The `cliningsmith_etal` file is the replication data behind the Clingingsmith,
Khwaja and Kremer study of the Hajj, the annual Muslim pilgrimage to Mecca. Far
more Pakistanis applied for a Hajj visa than there were visas, so a **lottery**
decided who received one. Among applicants, `success` = 1 marks the lottery
winners (who then made the pilgrimage) and `success` = 0 marks the losers.

Because the lottery is random *among applicants*, winners and losers are
comparable: the same mix of devotion, background, and motivation, differing only in
the luck of the draw. The `views` column is a composite index of how positively a
person rates people from other countries and ethnic groups, measured after the
pilgrimage season. A higher score means warmer views toward outsiders. The question
the lottery lets you ask honestly: does making the pilgrimage change those views?

### 🔮 Pause & Predict

You will compute the simple difference in the `views` index between lottery winners
and losers. Because the lottery is as-if random, that difference carries a causal
reading. Before running the next cell, commit to two things. Does making the
pilgrimage *raise* or *lower* the average views-toward-outsiders score? And will
the 95% interval exclude zero, or overlap it? Write a direction and a yes/no, with
one sentence of reasoning. Do not peek first.

### YOUR ANSWER HERE:

**Direction of the effect (up / down / no change):**

**Will the 95% interval exclude zero? (yes / no) and why:**

---

### 🛠️ Hands-On: analyze the lottery as a difference in means

Run the cell below. It loads the lottery data, confirms its shape, then computes the
difference in the `views` index between winners and losers, its standard error, and
a 95% interval. This is the classroom-simple, unadjusted analysis, presented with
its lottery-as-randomization argument. Read the numbers before the next markdown
cell.

**🔴 Live in class: we run this one together.**
**Before you ask:** write one sentence predicting the sign of the difference and
whether its interval clears zero (you committed a direction last cell; now commit
the interval).

> 💡 **Gemini Prompt:** "This cell computes the simple unadjusted difference in a
> 'views toward other groups' index between people who WON a Hajj visa lottery and
> those who LOST, among applicants, with a 95% interval: [paste the next cell].
> Explain why a lottery *among applicants* makes winners and losers comparable, so a
> plain difference in means carries a causal reading. Then explain why that same
> fact limits the claim to *applicants* and not to everyone. Finally, give me one
> independent way to confirm the difference and interval the cell printed."
>
> **After running, verify (counters *silent scope change*: 'the pilgrimage changes
> views' is a broader claim than 'winning the lottery changed applicants' views'):**
> - [ ] Confirm the difference and interval Gemini discusses match your printout
>   exactly: its sign, its size, and whether the interval clears zero, not numbers it
>   guessed.
> - [ ] Check that its claim boundary says *applicants*, not everyone. The lottery
>   randomizes only among people who applied, so people who never applied are a
>   different group. Reject any sentence that generalizes past applicants.
> - [ ] Run its independent check yourself. If it cannot produce one that runs, keep
>   your printout.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# Load the natural experiment and confirm its shape before trusting anything.
hajj = load_course_data("cliningsmith_etal.csv")
assert hajj.shape == (958, 8), "unexpected shape — flag this!"
print("✓ Loaded cliningsmith_etal.csv —", hajj.shape[0], "rows,", hajj.shape[1], "columns")
print("  success = 1 won the visa lottery (made the pilgrimage); 0 lost.")
print()

# Group sizes and the views index by lottery outcome.
by_outcome = hajj.groupby("success")["views"].agg(n="count", mean="mean")
by_outcome.index = ["lost lottery (success=0)", "won lottery (success=1)"]
print("Views-toward-other-groups index by lottery outcome:")
print(by_outcome.round(3))
print()

# Difference in means + standard error + 95% interval.
won = hajj.loc[hajj.success == 1, "views"]
lost = hajj.loc[hajj.success == 0, "views"]
diff = won.mean() - lost.mean()
se = np.sqrt(won.var(ddof=1) / len(won) + lost.var(ddof=1) / len(lost))
ci_lo, ci_hi = diff - 1.96 * se, diff + 1.96 * se
print(f"difference (won - lost): {diff:+.3f}   standard error: {se:.3f}")
print(f"95% interval: [{ci_lo:+.3f}, {ci_hi:+.3f}]   (excludes 0: {ci_lo > 0})")

In [ ]:
# Self-check: direction and interval match what we will claim.
assert len(won) == 510 and len(lost) == 448, "arm sizes changed — investigate"
assert diff > 0, "winners should score higher — report the sign honestly either way"
assert ci_lo > 0, "the 95% interval should exclude zero"
print(f"✓ Self-check passed: winners n={len(won)}, losers n={len(lost)}.")
print(f"✓ Winners score {diff:+.2f} higher on the views index, "
      f"95% interval [{ci_lo:+.2f}, {ci_hi:+.2f}], excluding zero.")
print("✓ This is the simple UNADJUSTED difference — the classroom version of the study.")

In [ ]:
# Show the two group means with the difference marked (winners vs losers).
fig, ax = plt.subplots(figsize=(7.5, 5))
means = [lost.mean(), won.mean()]
bars = ax.bar(["Lost lottery\n(control)", "Won lottery\n(pilgrimage)"], means,
              color=["#4C72B0", "#55A868"], edgecolor="white")
ax.set_ylabel("Views-toward-other-groups index")
ax.set_title(f"The as-if-random lottery: winners score {diff:+.2f} higher on views")
for b, v in zip(bars, means):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.03, f"{v:.2f}", ha="center", fontsize=11)
ax.set_ylim(0, max(means) + 0.5)
plt.tight_layout(); plt.show()
print("The gap between the bars is the lottery's causal estimate, for applicants.")

**Reading the evidence.** Winners score about +0.47 higher on the views index than
losers, and the 95% interval, roughly [+0.16, +0.79], stays above zero. Because the
lottery is as-if random among applicants, that gap carries a causal reading:
winning the lottery, and thus making the pilgrimage, *raised* warmth toward people
from other groups, among applicants. No confounder had to be measured, because the
lottery closed every back door at once. That is the power of a natural experiment.

Read the boundary as carefully as the result. The lottery randomizes only *among
people who applied*, so the claim covers applicants, not everyone. And this is the
simple unadjusted difference, the classroom version, not the study's full estimate.
The number says an effect exists and points up. It does not say *why* the
pilgrimage warmed views, and it does not speak for people who never applied.

### 🔍 Reading the Evidence: what the lottery lets you claim

In the cell below, answer three things about the +0.47 difference. First: write the
**most precise causal claim** the lottery permits, worded to name *whom* it covers
(careful: the lottery randomizes among applicants, not among all people). Second:
name two things this difference does **not** establish, no matter how clean the
lottery. Third: name the crossing you would have to pay for to carry this effect to
*all* Pakistanis, not just applicants, and say what would license it.

### YOUR ANSWER HERE:

**The most precise causal claim the lottery permits (and over whom):**

**Two things it does NOT establish:**

**The crossing to carry it beyond applicants, and what would license it:**

---

## 6. Three Arguments, Three Untestable Assumptions

**Guiding question:** *when the world does not hand you a clean lottery, what other
as-if-random moments can you exploit, and what must you take on faith?*

The lottery was the easy case. Most natural experiments are messier, and
researchers lean on three named arguments. Each is an **identification argument**,
your written reason that a comparison recovers a causal effect, not a formula you
just apply. And each rests on one **untestable assumption**: a claim the design
depends on that the data can never confirm, only make plausible.

- **Difference-in-differences (DiD)**: compare the *change over time* in a treated
  group to the change in an untreated group; the treated group's *extra* change is
  the effect. Example: one state raises its minimum wage and a neighbor does not,
  so you compare each state's employment change across the same years. **Untestable
  assumption, parallel trends:** absent the treatment, the two groups would have
  moved in parallel. The data show the past, never the counterfactual future the
  treated group did not get.
- **Regression discontinuity (RDD)**: treatment switches on at a sharp **cutoff** of
  some score, so units just below and just above are nearly identical except for
  treatment, and the *jump* in the outcome at the cutoff is the effect. Example: a
  scholarship goes to applicants at or above a test score, so you compare those just
  below against those just above. **Untestable assumption, as-if random at the
  cutoff:** nothing else changes at the line and units cannot precisely nudge their
  own score across it.
- **Instrumental variables (IV)**: use a **nudge** (an instrument) that pushes some
  units toward treatment but reaches the outcome *only through* treatment. Example:
  election-day rainfall nudges some people to stay home, so you use rainfall to
  study how turnout affects a result. **Untestable assumption, exclusion:** the
  nudge touches the outcome through treatment alone, with no back door of its own.

You cannot prove any of these three assumptions from the data. You can only argue
them well and let a skeptic attack. That is the honest character of observational
causal work, and it leads straight to the failure that catches earnest
researchers.

**Design mimicry: the vocabulary is not the argument.** The most common causal
failure among sincere researchers is **design mimicry**: borrowing the vocabulary,
"difference-in-differences," "natural experiment," to decorate a comparison whose
assumption was never argued. Writing "DiD" does not make parallel trends true.
Calling something a "natural experiment" does not make the assignment as-if random.
The word *because* is earned by the argued and probed assumption, never by the name
of the method. A skeptic reads past the label straight to the assumption, and so
should you.

That gives you the line this whole week patrols.

- **Causal-language boundary**: the point past which your evidence stops earning the
  word *because* and must switch to *associated with*. Example: "coffee is
  associated with longer life" is honest from observational data with an unmeasured
  confounder; "coffee lengthens life" is not, unless you identified the effect. A
  finding that keeps causal language it did not earn is over the line.

> **A question that often comes up here:** *"If DiD, RDD, and IV can reach causal
> answers without an experiment, why randomize at all?"* Because each buys its
> causal reading with an assumption a skeptic can attack: parallel trends, a clean
> unmanipulated cutoff, a nudge with no back door. Randomization makes the
> counterfactual valid *by construction*, while these tools make it valid *by
> argument*, and arguments can be wrong. When a real experiment is available, you
> take it. Next week is about exactly that.

---

### ⏸ In class through here

**You have reached the end of Lecture 2's in-class block.** Everything below feeds
your M5 deliverable. Finish it before Friday's async studio: run the three
intuition figures, adapt the skeptic prompt to your own project, interrogate what
comes back, make the AI-free identification decision, do the practice, and draft
your M5 spine. The 🧑‍⚖️ Human-Only Checkpoint is the never-delegate core of M5, so
give it real time.

---

### The three arguments, drawn in worlds you can run

Reading about parallel trends and cutoffs is one thing. Watching each argument
succeed in a world built to satisfy it teaches you what "it worked" looks like, and
therefore what a violation would look like in real data. The three cells below draw
a DiD world, an RDD world, and an IV diagram. Run them, then use the prompt to pin
each design's one untestable assumption.

**🏠 Homework depth: run this on your own.**
**Before you ask:** for each of the three designs, write your own one-sentence guess
at the single assumption a skeptic would attack. Commit before the tool answers.

> 💡 **Gemini Prompt:** "For difference-in-differences, regression discontinuity, and
> instrumental variables, state the ONE untestable assumption each design rests on,
> in a table with columns for the design, the assumption, and what a violation would
> look like. Then take this scenario for each and judge whether the assumption
> plausibly holds: [paste a one-line scenario of your own for each design]. Finally,
> argue against your own judgment as a hostile reviewer would."
>
> **After running, verify (counters *plausible-but-wrong-method*: a design label can
> be attached to a comparison whose key assumption plainly fails):**
> - [ ] For each design, confirm the assumption it names is the real load-bearing one
>   (parallel trends for DiD, as-if-random-at-the-cutoff for RDD, exclusion for IV),
>   not a minor side condition.
> - [ ] Check its verdict on your scenario against the figures below: does the DiD
>   post-treatment gap really match the built-in effect, and does the RDD local jump
>   recover the planted one? If its story ignores the printed numbers, distrust it.
> - [ ] Decide *yourself* whether each assumption holds in your own project. A label
>   the tool endorses is not an argument you have made.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

In [ ]:
# ILLUSTRATION 1 (simulated, not data): a parallel-trends / DiD world.
did_rng = np.random.default_rng(SEED)
time = np.array([0, 1, 2, 3])
post = time >= 2                       # treatment turns on between period 1 and 2
control = 40 + 3 * time + did_rng.normal(0, 0.3, 4)
did_effect = 6                         # the built-in causal effect
treated_cf = control + 8               # counterfactual: parallel to control, shifted up
treated = treated_cf + did_effect * post  # observed = counterfactual PLUS the effect, post-treatment

fig, ax = plt.subplots()
ax.plot(time, control, "o-", color="#4C72B0", label="control group")
ax.plot(time, treated, "s-", color="#55A868", label="treated group (observed)")
ax.plot(time, treated_cf, "s--", color="#55A868", alpha=0.4,
        label="treated counterfactual (parallel, no treatment)")
ax.axvline(1.5, color="gray", linestyle=":", label="treatment turns on")
ax.set_xlabel("time period"); ax.set_ylabel("outcome")
ax.set_title("ILLUSTRATION: difference-in-differences\nthe gap the treated line opens above its parallel counterfactual IS the effect")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

post_gap = (treated - treated_cf)[post]
print(f"Built-in effect = {did_effect}. Post-treatment gap (treated minus its "
      f"counterfactual) = {np.round(post_gap, 2).tolist()}.")
print("The picture recovers the effect ONLY because the world was built with parallel trends.")

In [ ]:
# ILLUSTRATION 2 (simulated, not data): a threshold / RDD world.
rdd_rng = np.random.default_rng(SEED)
x = rdd_rng.uniform(0, 100, 300)       # a running score (e.g., an eligibility test)
rdd_cutoff = 50
rdd_jump = 8                           # the built-in effect: a jump at the cutoff
y = 20 + 0.3 * x + rdd_jump * (x >= rdd_cutoff) + rdd_rng.normal(0, 3, 300)

fig, ax = plt.subplots()
ax.scatter(x[x < rdd_cutoff], y[x < rdd_cutoff], s=14, color="#4C72B0",
           label="below cutoff (untreated)")
ax.scatter(x[x >= rdd_cutoff], y[x >= rdd_cutoff], s=14, color="#55A868",
           label="at/above cutoff (treated)")
ax.axvline(rdd_cutoff, color="gray", linestyle=":", label=f"cutoff = {rdd_cutoff}")
ax.set_xlabel("running score"); ax.set_ylabel("outcome")
ax.set_title("ILLUSTRATION: regression discontinuity\nthe vertical cliff at the cutoff is the effect")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

# Compare ONLY points just below vs just above the cutoff (a local comparison).
lo = (x >= 44) & (x < 50)
hi = (x >= 50) & (x < 56)
local_jump = y[hi].mean() - y[lo].mean()
print(f"Built-in jump = {rdd_jump}. Local jump (just above minus just below the "
      f"cutoff) = {local_jump:.1f}.")
print("It recovers the planted jump up to the noise in a thin window near the line.")

In [ ]:
# Self-check: the DiD gap equals the built-in effect; the RDD local jump recovers it.
assert list(np.round(post_gap, 2)) == [6.0, 6.0], "DiD gap should equal the built-in effect of 6"
assert 6 < local_jump < 13, "RDD local jump should land near the built-in 8 (window noise expected)"
print(f"✓ Self-check passed: DiD recovers the built-in +6; RDD's local comparison "
      f"recovers about {local_jump:.0f} (planted +8).")

In [ ]:
# ILLUSTRATION 3: an instrumental-variables diagram (a picture, no estimation).
# Rainfall (instrument) -> Turnout (treatment) -> Result (outcome); a confounder U
# feeds turnout and result. The instrument's power: it has NO direct path to Result.
IV = nx.DiGraph()
pos = {"Rainfall\n(instrument)": (0.0, 0.0), "Turnout\n(treatment)": (1.6, 0.0),
       "Result\n(outcome)": (3.2, 0.0), "Confounder\nU": (2.4, 1.3)}
iv_edges = [("Rainfall\n(instrument)", "Turnout\n(treatment)"),
            ("Turnout\n(treatment)", "Result\n(outcome)"),
            ("Confounder\nU", "Turnout\n(treatment)"),
            ("Confounder\nU", "Result\n(outcome)")]
IV.add_edges_from(iv_edges)
fig, ax = plt.subplots(figsize=(9, 4))
node_colors = ["#F2C9A0" if n == "Confounder\nU" else "#DCE6F1" for n in IV.nodes]
nx.draw_networkx_nodes(IV, pos, node_color=node_colors, edgecolors="#333333",
                       linewidths=1.5, node_size=5200, ax=ax)
nx.draw_networkx_labels(IV, pos, font_size=8, ax=ax)
edge_cols = ["#4C72B0", "#4C72B0", "#DD8452", "#DD8452"]
for e, c in zip(iv_edges, edge_cols):
    nx.draw_networkx_edges(IV, pos, edgelist=[e], edge_color=c, arrowstyle="-|>",
                           arrowsize=18, width=2.2, node_size=5200, ax=ax)
nx.draw_networkx_edge_labels(IV, pos, font_size=7.5, edge_labels={
    iv_edges[0]: "relevance", iv_edges[1]: "the effect",
    iv_edges[2]: "back door", iv_edges[3]: "back door"}, ax=ax)
ax.text(1.6, -0.9, "exclusion assumption: NO arrow Rainfall -> Result\n(the instrument reaches the outcome only through turnout)",
        ha="center", fontsize=8, color="#555555")
ax.set_title("ILLUSTRATION: instrumental variables — a nudge with no back door of its own")
ax.set_ylim(-1.2, 1.8); ax.axis("off")
plt.tight_layout(); plt.show()
print("✓ IV diagram drawn. The instrument is useful only if the dashed 'exclusion'")
print("  claim holds: rainfall touches the result through turnout and nowhere else.")

**Reading the three pictures.** Each figure shows an argument *succeeding* because
the world was built to satisfy it. DiD recovers the planted +6 because the groups
were built to move in parallel. RDD recovers close to the planted +8 (about +9 here,
the wobble expected in a thin window) because nothing else jumps at the cutoff. IV
would work because the instrument was built with no
back door. In real data you get none of these guarantees for free. You get an
assumption you must argue and a skeptic who will attack it. The figures teach you
the shape of success so you can recognize the shape of failure.

### 📝 Practice: match the design, then name the first attack

*(Human-first: reason through all three yourself before any AI. Naming the attack is
the graded skill.)*

Match each scenario to its identification argument (difference-in-differences,
regression discontinuity, or instrumental variables, each used once), and write the
**one sentence a skeptic would attack first**, the untestable assumption the whole
claim rests on. Answer in the cell that follows.

- **A.** "One state raised its minimum wage in one year and a neighbor did not; we
  compare each state's change in employment across the surrounding years."
- **B.** "A merit scholarship goes to applicants at or above a test score of 1200;
  we compare college enrollment for applicants just below versus just above 1200."
- **C.** "Rainfall on election day nudges some people to stay home; we use
  election-day rainfall to study how turnout affects which candidate wins.

### YOUR ANSWER HERE:

**A (design / first attack):**

**B (design / first attack):**

**C (design / first attack):**

---

### Red-team a causal sentence you wrote

You have an honest result from the lottery. Now practice defending its wording. In
the cell below, first write one sentence stating the lottery's causal finding,
worded to stop exactly where the evidence stops (an effect on *applicants*, of about
+0.47, with its interval, and no claim about mechanism or everyone). Write it
yourself. Then make Gemini try to break it.

**🏠 Homework depth: run this on your own.**
**Before you ask:** paste the sentence you wrote above into the prompt. Do not let
Gemini draft the sentence. You wrote it; the tool only attacks it.

> 💡 **Gemini Prompt:** "Here is a causal sentence I wrote about a visa-lottery
> natural experiment: '[paste your sentence].' The effect is about +0.47 on a views
> index, 95% interval roughly [+0.16, +0.79], among applicants only, from a simple
> unadjusted difference. Act as a hostile peer reviewer. Name every place my sentence
> over-reaches, hides a limitation, or claims more than an unadjusted applicant-only
> estimate can. Do not rewrite it for me. List the specific weaknesses so I can fix
> them myself."
>
> **After running, verify (counters *sycophantic agreement*: an AI that praises the
> sentence you wrote has reviewed your ego, not your evidence):**
> - [ ] If every objection is mild, or it calls your sentence "well-balanced," push
>   back: "assume it is wrong; name the single worst flaw." Weak praise is a red flag.
> - [ ] Check each objection against your printout: the effect is about +0.47 and
>   *positive*, among applicants, so reject any objection that misstates the sign or
>   forgets the applicant-only boundary.
> - [ ] Log this use in your AI Research Ledger: task, tool, decision, verification.

### YOUR ANSWER HERE:

**My causal sentence about the lottery (I wrote this, not Gemini):**

**The weaknesses Gemini raised, and which ones I accepted or rejected, and why:**

---

### 🔁 Modify the Prompt

Above, Gemini attacked a sentence about the *lottery*. Now aim the same attack at
*your own project*. First, in one or two sentences, write your project's causal
claim (or your honest causal-language boundary) yourself. Do not let AI draft it.
Then adapt the attack prompt so Gemini red-teams *your* identification argument.

> **Base prompt (the attack, from above):** "Here is a causal claim I wrote: '[your
> claim].' Act as a hostile peer reviewer and name every place it over-reaches or
> claims more than my design identifies. Do not rewrite it for me."

Rewrite it so it names *your* treatment, *your* outcome, and the one untestable
assumption *your* design rests on (selection on observables, parallel trends,
as-if-random-at-a-cutoff, or exclusion). In the cell below: paste your claim, paste
your adapted attack prompt, predict the single strongest objection Gemini will
raise, then run it and note whether your prediction held.

### YOUR ANSWER HERE:

**My project's causal claim or causal-language boundary (AI did not draft this):**

**My adapted attack prompt (my treatment, my outcome, my untestable assumption):**

**The strongest objection I predict Gemini will raise:**

**What Gemini actually raised, and how I fixed my claim:**

---

### 🔬 Interrogate the Output

Gemini just handed you objections to your identification argument. Do not accept the
attack as automatically right either. Interrogate it against your design, using four
checks. Answer each in the cell below.

- **Claims:** does each objection point to a real threat to *your* counterfactual,
  and does it match how treatment was actually assigned in your design? An objection
  about a confounder you already ruled out with a lottery or a cutoff is not fair.
- **Assumptions:** what does the attack assume about your assignment mechanism? Did
  the tool invent a back door your design does not actually have, because it cannot
  see how you assigned treatment?
- **Missing information:** which real threat did the attack *miss*? A reviewer that
  skips your one untestable assumption left the most important objection on the table.
- **Overstatements:** does any objection sound more certain than the evidence
  supports? Flag the exact words.

And one rule that governs the whole notebook: **code running without errors is not
the same as code being correct.** A cell can execute cleanly and still compute the
wrong quantity, and a fluent attack can still be wrong about your design. Verify the
claim against your assignment mechanism, not against the confidence of the prose.

### YOUR ANSWER HERE:

**Claims (does each objection match how my treatment was assigned?):**

**Assumptions (did the attack invent a back door my design does not have?):**

**Missing information (the real threat the attack skipped):**

**Overstatements (exact words):**

---

### 🧑‍⚖️ Human-Only Checkpoint

Close Gemini for this one. No AI, no notebook search. This is a never-delegate
decision: whether your project earns a causal claim is yours to declare and defend.
In the cell below, write, in your own words, the core of your M5 deliverable.

1. Your project's **causal question**, stated as a treatment and an outcome.
2. **Either** an **identification argument** (three sentences: the estimand you
   want; why your comparison is a valid counterfactual, naming the design and its
   one untestable assumption; what therefore follows, with uncertainty), **or** an
   honest **causal-language boundary** (one sentence: "my design does not identify
   an effect because [the confounder I cannot rule out], so my claim stops at
   *associated with*").

If you cannot write the identification argument honestly yet, that is a finding, not
a failure. It tells you which assumption you still have to defend, or that your claim
belongs on the *associated with* side of the line.

### YOUR ANSWER HERE:

**My causal question (treatment and outcome):**

**EITHER my identification argument (estimand / valid counterfactual + design + untestable assumption / therefore, with uncertainty):**

**OR my honest causal-language boundary (why I stop at 'associated with'):**

---

### 🎯 Project Transfer: the spine of your M5 deliverable

*(Human-first: draft every piece yourself. AI may critique it after, but the strategy
you submit has to be reasoned by you.)*

This is where the week becomes your **M5 causal identification strategy (or honest
causal-language boundary)**. In the cell below, draft its spine for *your* project:

1. **The causal question:** your treatment, your outcome, and the estimand (the
   average effect you are after), one line each.
2. **The causal diagram:** in words or a quick sketch, the arrows among treatment,
   outcome, and the one confounder you most fear opens a back door.
3. **The identification argument, or its honest absence:** name your leverage
   (selection on observables, a natural experiment, DiD, RDD, or IV), state the one
   untestable assumption it rests on, and how a skeptic would attack it. If you have
   no leverage, write the causal-language boundary instead.
4. **One recompute:** the single number you would recompute yourself to check any AI
   or collaborator's estimate, and how. A **recompute** is an independent redo of a
   number by a second route, for example rebuilding the lottery difference by
   hand-filtering winners and losers and subtracting their means, rather than just
   rerunning the same cell.

These four pieces are what the required 'Causal Identification Skeptic' review at
Friday's studio will attack. Write them so a reader who has never seen your project
could find the exact assumption your causal claim rests on.

### YOUR ANSWER HERE:

**1. The causal question (treatment / outcome / estimand):**

**2. The causal diagram (arrows + the confounder I most fear):**

**3. The identification argument, or its honest absence (leverage / untestable assumption / the skeptic's attack):**

**4. One recompute (the number I would check myself, and how):**

---

### 📒 AI Research Ledger

Log every AI use from this notebook in the ledger. One worked row is filled in as a
model, and it captures the discipline this week teaches: **you wrote the causal
claim, the AI attacked it.** This notebook offers five prompts, one live per lecture
and three homework-depth, so your ledger carries a row for each one you actually ran.
The ledger is a graded habit, not paperwork: it is how you show your causal claim was
probed, not just asserted.

| Task delegated | Tool used | Prompt | Output summary | Decision | Verification method | Remaining concern | Responsible researcher |
|---|---|---|---|---|---|---|---|
| Red-team the causal sentence I wrote about the Hajj lottery | Gemini | "Here is a causal sentence I wrote: [sentence]. Act as a hostile reviewer; name every over-reach; do not rewrite it." | Four objections: one fair (I did not flag that this is the unadjusted, not the study's full, estimate), three that misread the applicant-only boundary as a flaw | Accepted the fair objection and added the unadjusted caveat; rejected the three that misread the boundary | Rechecked the printout: effect +0.47, positive, applicants only; kept only objections consistent with those numbers | A real design will not hand me a clean lottery to lean on | *(your name)* |
|  |  |  |  |  |  |  |  |
|  |  |  |  |  |  |  |  |

---

### 🛡️ Exit Defense

Defense #05 — write, in your own words:

1. **The claim I can defend:** one bounded sentence, drawn from today's data or your
   own project, that you would put your name on.
2. **Its boundary:** what this evidence does NOT establish. Name the compass position
   (causal reasoning) and whether you identified the effect (and by what argument) or
   honestly stopped at *associated with*.
3. **My uncertainty and limitations:** one sentence naming your untestable assumption,
   an unmeasured confounder, or the applicant-only / unadjusted caveat.
4. **AI check:** what you delegated to Gemini, whether you **accepted, changed, or
   rejected** what it returned, and the specific check that decided it (a number you
   recomputed, a boundary you confirmed, an assumption you tested).

### YOUR ANSWER HERE:

**1. The claim I can defend:**

**2. Its boundary (identified by ___ , or stops at 'associated with'):**

**3. My uncertainty and limitations:**

**4. AI check (what I delegated, how I verified):**

---

## 7. Wrap-Up

Across two lectures you learned what it takes to earn the word *because* without an
experiment. You met the **counterfactual**, the missing world you never see for one
person, and the **estimand**, the average effect you are actually after. You drew
the **confounder**'s back door and watched it inflate a naive comparison to three
times the truth, then saw that adjusting for an *observed* confounder recovers the
effect, while an *unobserved* one leaves you stuck. Then you left adjustment behind
for the world's own experiments: a real visa lottery that identified a causal effect
with no confounder measured, and the three arguments (parallel lines, a cutoff, a
nudge) that each buy a causal reading with one untestable assumption a skeptic
attacks first.

> **"The word 'because' is earned by a counterfactual you can defend, never by the
> name of a method. Adjusting for the confounders you happen to see is not
> identification, and a design label is not an argument. When you cannot defend the
> counterfactual, the honest claim stops at 'associated with.'"**

Next the course turns to the experiment itself, where random assignment makes the
counterfactual valid by construction rather than by argument. Your M5 causal
identification strategy, or its honest causal-language boundary, submits at Friday's
async studio, where the required 'Causal Identification Skeptic' review attacks the
one assumption your claim rests on. Bring your causal diagram and your identification
argument. They are about to meet a skeptic you cannot argue back to in real time.

---

## 8. Sources & Provenance

**Provenance of this notebook:**
- *RDSS ch. 16 'Observational: causal' | counterfactuals, confounding, selection-on-observables, DiD/IV/RDD as identification arguments, natural experiments, causal-language boundaries | adapted (concept-level, honors non-quant audience)*
- *RDSS ch. 6-7 (model/inquiry) | potential outcomes Y(1)/Y(0), the estimand, the reach note (SATE vs PATE) | adapted*
- *DeclareDesign observational-causal design library | the observational causal pathway framing | referenced*
- *cliningsmith_etal.csv | rdss package data | Clingingsmith, Khwaja & Kremer Hajj-lottery study replication; simple unadjusted difference in means with an interval | adapted (the classroom-simple version, not the study's full estimate)*
- *fresh | the hidden-confounder potential-outcomes simulation + the two causal diagrams + the DiD/RDD/IV intuition figures | newly-constructed-from-source-concept*
- *callingbullshit.org (public index) | correlation-and-causation material | linked as optional study, index pages only | referenced*

**Dataset attribution:** Dataset from the `rdss` package (Blair, Coppock &
Humphreys, MIT License), companion to *Research Design in the Social Sciences*
(2023).

**Readings:**
- Blair, G., Coppock, A., & Humphreys, M. (2023). *Research Design in the Social
  Sciences*, ch. 16 'Observational: causal' and ch. 6-7 (model, inquiry, potential
  outcomes). Free online: [book.declaredesign.org](https://book.declaredesign.org/).
- *(Optional)* Bergstrom, C. & West, J. *Calling Bullshit*, correlation-and-causation
  case index: [callingbullshit.org/tools](https://callingbullshit.org/tools/).

---

<center>

Thank you!

</center>